# <center> <img src="figs/LogoUFSCar.jpg" alt="Logo UFScar" width="110" align="left"/>  <br/> <center>Universidade Federal de S?o Carlos (UFSCar)<br/><font size="4"> Departamento de Computa??o, campus Sorocaba</center></font>
</p>

<br/>
<font size="4"><center><b>Disciplina: Aprendizado de M?quina</b></center></font>
  
<font size="3"><center>Prof. Dr. Tiago A. Almeida</center></font>

<br/>
<br/>

<center><i><b>
Aten??o: n?o s?o autorizadas c?pias, divulga??es ou qualquer tipo de uso deste material sem o consentimento pr?vio dos autores.
</center></i></b>

# <center>Exerc?cio - Sistemas de Recomenda??o</center>

Introdu??o
----------
Neste *notebook*, voc? far? um protocolo experimental completo no contexto de Sistemas de Recomenda??o, aplicando dois algoritmos de Filtragem Colaborativa, KNN e SVD, sobre uma base real de recomenda??o de filmes. Os experimentos que ser?o apresentados foram projetados com o intuito de facilitar o entendimento da ?rea e serem gen?ricos, permitindo sua reprodu??o em outros dom?nios de aplica??o.

Antes de come?ar, ? recomend?vel que voc? revise os conceitos apresentados em aula.


## Instru??es
Este arquivo cont?m o c?digo que auxiliar? no desenvolvimento do exerc?cio. Voc? precisar? completar as seguintes fun??es:

* <tt>knn()</tt>
* <tt>svd_sgd_optimizer()</tt>
* <tt>rmse_mae()</tt>
* <tt>precision_recall_f1()</tt>
* <tt>ndcg()</tt>

## Parte 1: Leitura e prepara??o da base de dados

A base de dados utilizada nesse trabalho foi extra?da do site [movielens.org](https://movielens.org). 

O **MovieLens** ? um portal de recomenda??o de filmes gerenciado pelo grupo de pesquisa GroupLens, da Universidade de Minnesota. Diversas varia??es da base de dados encontram-se hospedadas no [site do grupo](https://grouplens.org/datasets/movielens/). Ao longo da disciplina, utilizaremos a **ml-latest-small**, vers?o reduzida, destinada para estudo.

A base original possui diversos arquivos diferentes, seguindo um determinado padr?o. Nem todos ser?o necess?rios para o trabalho. Iremos ent?o formatar a base, selecionando apenas as informa??es relevantes. Adicionalmente, usaremos apenas 20% da base, com o intuito de reduzir o tempo de execu??o dos experimentos presentes no notebook.

Um c?digo para preprocessamento e formata??o da base encontra-se na fun??o `format_movielens_dataset()`, presente no arquivo `recsys_utils.py` enviado junto com este trabalho. Iremos execut?-lo.

In [ ]:
# -*- coding: utf-8 -*-

# Caminho dos arquivos
FILES_DIRECTORY = "ml-100k"

from recsys_utils import format_movielens_dataset

format_movielens_dataset(raw_dataset_folder=FILES_DIRECTORY, sampling_rate=0.2)
print()

? fortemente encorajado que voc? olhe o c?digo para entender o que foi realizado. Ap?s sua chamada, **3** novos arquivos foram criados, dentro da pasta <tt>datasets</tt>. S?o eles:
* **users.csv**: contendo informa??es sobre os usu?rios do sistema;
* **items.csv**: contendo informa??es sobre os filmes do sistema;
* **interactions.csv**: contendo todas as notas que os usu?rios deram aos filmes.

Vamos carregar os arquivos como objetos `pandas.DataFrame` e visualiz?-los.

In [ ]:
import csv
import pandas as pd

# Carrega bases de dados
users = pd.read_csv('dataset/users.csv', sep=';', quotechar='"', quoting=csv.QUOTE_ALL, encoding='latin-1', header=0, index_col=None)
items = pd.read_csv('dataset/items.csv', sep=';', quotechar='"', quoting=csv.QUOTE_ALL, encoding='latin-1', header=0, index_col=None)
interactions = pd.read_csv('dataset/interactions.csv', sep=';', quotechar='"', quoting=csv.QUOTE_ALL, encoding='latin-1', header=0, index_col=None)

# Preenche generos nulos
items['genero'] = items['genero'].fillna('')

# Exibe as 5 primeiras linhas de cada arquivo
print('\nUsu?rios:')
display(users.head(5))
print('\nItens:')
display(items.head(5))
print('\nIntera??es:')
display(interactions.head(5))

# Faz a contagem de usuarios, itens e interacoes
n_users = len(users)
n_items = len(items)
n_interactions = len(interactions)
print('Temos {} usu?rios'.format(len(users)))
print('Temos {} itens'.format(len(items)))
print('Temos {} intera??es\n'.format(len(interactions)))

Com a base de dados carregada em mem?ria, podemos ir para a segunda etapa.

---
## Parte 2: Divis?o da base em Treinamento, Valida??o e Teste

Para treinar, ajustar os par?metros e avaliar nossos modelos, iremos separar a base de dados usando _holdout_, com **80% dos dados para treinamento**, **10% para valida??o** e **10% para teste**. Para isso, usaremos a fun??o `train_test_split()` da biblioteca `sklearn`. A fun??o s? consegue separar a base de dados em duas novas, n?o conseguindo assim separar em treino, valida??o e teste com uma ?nica chamada. Iremos ent?o extrair primeiramente as amostras de treinamento, separando o restante entre valida??o e teste.

In [ ]:
from sklearn.model_selection import train_test_split

interactions_train, remaining = train_test_split(
    interactions,
    train_size=0.8,
    test_size=0.2,
    shuffle=False
)
interactions_train = interactions_train.copy() # Apenas para evitar mensagens de warning adiante

interactions_val, interactions_test = train_test_split(
    remaining,
    train_size=0.5,
    test_size=0.5,
    shuffle=False
)
interactions_val = interactions_val.copy() # Apenas para evitar mensagens de warning adiante
interactions_test = interactions_test.copy() # Apenas para evitar mensagens de warning adiante

### Remo??o do problema de Cold-Start na valida??o e teste

Ap?s a separa??o em treino-valida??o-teste, pode acontecer de clientes ou itens ficarem presentes apenas nas bases de teste, e ausentes na de treino. Esse ? o problema do **cold-start**, ou partida fria, que deve ser abordado por algoritmos espec?ficos (como baseados em conte?do).

Como este problema foge do escopo do notebook, vamos remover todos os casos de cold-start:

In [ ]:
interactions_val = interactions_val[
    (interactions_val['id_usuario'].isin(interactions_train['id_usuario']))
    &(interactions_val['id_item'].isin(interactions_train['id_item']))
].copy()

interactions_test = interactions_test[
    (interactions_test['id_usuario'].isin(interactions_train['id_usuario']))
    &(interactions_test['id_item'].isin(interactions_train['id_item']))
].copy()

items = items[items['id_item'].isin(interactions_train['id_item'])]
users = users[users['id_usuario'].isin(interactions_train['id_usuario'])]

print('Nossa base de TREINAMENTO cont?m {} intera??es\n'.format(len(interactions_train)))
print('Nossa base de VALIDA??O cont?m {} intera??es\n'.format(len(interactions_val)))
print('Nossa base de TESTE cont?m {} intera??es\n'.format(len(interactions_test)))

### Codifica IDs dos usu?rios e items para ficar no intervalo de 0 a N

Para simplificar as implementa??es seguintes, iremos alterar os IDs dos usu?rios e itens, para que todos passem a variar no intervalo 0 a "qtde. de usu?rios" / "qtde. de itens". Com essa altera??o, ser? mais f?cil trabalhar com os dados em formato matricial, de forma que uma linhda de ?ndice _i_ corresponda ao item _i_, por exemplo

In [ ]:
from sklearn.preprocessing import LabelEncoder

item_encoder = LabelEncoder()    
items.loc[:, 'id_item'] = item_encoder.fit_transform(items['id_item'].values)

user_encoder = LabelEncoder()
users.loc[:, 'id_usuario'] = user_encoder.fit_transform(users['id_usuario'].values)

interactions_train.loc[:, 'id_item'] = item_encoder.transform(interactions_train['id_item'].values)
interactions_train.loc[:, 'id_usuario'] = user_encoder.transform(interactions_train['id_usuario'].values)

interactions_val.loc[:, 'id_item'] = item_encoder.transform(interactions_val['id_item'].values)
interactions_val.loc[:, 'id_usuario'] = user_encoder.transform(interactions_val['id_usuario'].values)

interactions_test.loc[:, 'id_item'] = item_encoder.transform(interactions_test['id_item'].values)
interactions_test.loc[:, 'id_usuario'] = user_encoder.transform(interactions_test['id_usuario'].values)

---
## Parte 3: Implementa??o dos algoritmos de recomenda??o

Ser?o implementados dois algoritmos diferentes, que utilizam dados expl?citos para gerar a recomenda??o.
* **K-Vizinhos Mais Pr?ximos (KNN)**, baseado em vizinhan?a
* **Decomposi??o em Valores Singulares (SVD)**, baseado em fatora??o de matriz

### <center>**KNN**</center>

Para a implementa??o do KNN, primeiro iremos converter o DataFrame para uma matriz de intera??es, e ent?o iremos gerar a recomenda??o como visto em aula:

#### **Converter DataFrame para Matriz de Intera??es**

Antes de implementar as fun??es de predi??o de nota pelo KNN, ? necess?rio que nossos dados sejam representados como uma **Matriz de Intera??es**, onde cada linha corresponde a um item e cada coluna um usu?rio (ou o contr?rio). Com essa matriz, podemos recuperar os vetores de representa??o dos itens para o restante do algoritmo. Para isso, vamos transformar o dataframe de treino em uma matriz de intera??es:

In [ ]:
import numpy as np

interactions_matrix = np.zeros((n_items, n_users))
interactions_matrix[interactions_train['id_item'], interactions_train['id_usuario']] = interactions_train['nota'].values
print(interactions_matrix)

#### **Predi??o de nota atrav?s do KNN**

Com a matriz de intera??es, podemos realizar a predi??o das notas. A predi??o ? composta de algums etapas, que voc? dever? implementar:

* Calcular as similaridades entre os itens, o que pode ser feito usando `sklearn.metrics.pairwise.cosine_similarity`
* Percorrer cada um dos pares usu?rio-item que se deseja prever uma nota
* Encontrar os $k$ itens mais similares ao item alvo **dentre os itens que o usu?rio j? avaliou**
* Calcular a nota atrav?s da seguinte f?rmula:

$$R = \frac{\sum_{i=1}^{k}{(y_{u,i} \times s_i)}}{\sum_{i=1}^{k}{s_i}}$$

Na qual, $y_{u,i}$ corresponde a nota atribu?da pelo usu?rio ao item vizinho e $s_i$ representa a similaridade com o item alvo. Trata-se de uma simples m?dia ponderada.

Voc? dever? implementar a fun??o `knn()`, que recebe as notas dadas pelos usu?rios num formato de matriz de intera??es, os pares usu?rio-item que se deseja prever uma nota e um valor de $k$.

**Observa??es**
* Para calcular as similaridades entre item, pode-se utilizar `sklearn.metrics.pairwise.cosine_similarity`
* No caso de n?o ser poss?vel calcular uma nota (todos os itens avaliados pelo usu?rio tem similaridade 0.0 em rela??o ao item alvo), preencha com o valor padr?o **2.5**

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def knn(interactions_matrix, user_item_targets, k=10):
    """
    Preve as notas que os usuarios dariam para os itens usando o algoritmo de KNN
    
    ----------- Entrada -----------
       
    interactions_matrix: matriz np.array contendo as representacoes vetoriais de cada item por linha
    
    user_item_targets: np.array contendo os pares usuario-item para prever a nota, no qual a
        primeira coluna corresponde ao ID do usuario e a segunda coluna ao ID do item        
    
    k: numero de itens mais similares ao item alvo para observar
    
    ----------- Sa?da -----------
    
    item_sims: matriz contendo as similaridades entre items
    
    ratings: np.array de mesmo tamanho de user_item_targets contendo as notas previstas pelo KNN
    """
    
    # Inicializa a matriz de similaridades
    n_items, n_users = interactions_matrix.shape
    item_sims = np.zeros((n_items, n_items))
    
    # Inicializa as notas
    ratings = np.zeros(user_item_targets.shape[0])
    
    # Nota padrao
    DEFAULT_RATING = 2.5
        
    ###########################################################################
    ######################### COMPLETE O C?DIGO AQUI  #########################
    # Instru??es: voc? deve prever as notas para os pares usuario-item contidos
    #   em user_item_targets
    #
    # Para tal, calcula a similaridade entre os itens usando a funcao da 
    #   biblioteca sklearn cosine_similarity. Em seguida, para cada par
    #   usuario-item, voce deve calcular a nota prevista usando os K itens
    #   mais similares que o usuario ja avaliou
    
    item_sims = cosine_similarity(interactions_matrix)

    for i, (user, item) in enumerate(user_item_targets):
        interacted_items = np.where(interactions_matrix[:, user] > 0)[0]

        k_itens = np.argsort(item_sims[item, interacted_items])[-k:]

        ratings_top_k = interactions_matrix[interacted_items, user][k_itens]
        sim_top_k = item_sims[item, interacted_items][k_itens]

        if len(ratings_top_k) <= 0:
            ratings[i] = DEFAULT_RATING
        else:
            ratings[i] = np.dot(ratings_top_k, sim_top_k) / np.sum(sim_top_k)

    ###########################################################################
    
    return item_sims, ratings


# Verifica a implementacao
print("Se seu c?digo estiver correto, a nota prevista ser? 4.2184")
item_sims, pred_rating = knn(interactions_matrix, np.array([[398,829]]), k=3)
print("A nota prevista foi {:.4f}\n".format(pred_rating[0]))

print("Se seu c?digo estiver correto, a nota prevista ser? 3.4332")
item_sims, pred_rating = knn(interactions_matrix, np.array([[630,150]]), k=4)
print("A nota prevista foi {:.4f}\n".format(pred_rating[0]))

print("Se seu c?digo estiver correto, a nota prevista ser? 3.8214")
item_sims, pred_rating = knn(interactions_matrix, np.array([[723,863]]), k=10)
print("A nota prevista foi {:.4f}\n".format(pred_rating[0]))

print("Se seu c?digo estiver correto, a similaridade entre os itens 0 e 7 ser? 0.1687")
print("A similaridade ? {:.4f}\n".format(item_sims[0, 7]))

print("Se seu c?digo estiver correto, a similaridade entre os itens 101 e 368 ser? 0.4586")
print("A similaridade ? {:.4f}\n".format(item_sims[101, 368]))

print("Se seu c?digo estiver correto, a similaridade entre os itens 888 e 2 ser? 0.3588")
print("A similaridade ? {:.4f}\n".format(item_sims[888, 2]))

### <center> **SVD** </center>

O segundo m?todo ser? a t?cnica de fatora??o de matrizes chamada de SVD, ou _Singular Value Decomposition_.

Iremos otimizar o m?todo atrav?s da abordagem **Stochastic Gradient Descent**, em que ambos os vetores de fatores latentes s?o otimizados simultaneamente.

#### **Treinamento do SVD para ajuste de fatores latentes**

Voc? dever? implementar a fun??o `svd_sgd_optimizer()`, que recebe dois vetores de fatores latentes, de usu?rio e de item, assim como demais par?metros da otimiza??o e o valor-alvo (nota atribu?da), e deve calcular os novos vetores de fatores latentes minimizando o erro, **utilizando regulariza??o**.

Primeiro, voc? deve calcular a diferen?a entre o valor real e a predi??o:

$$e_{u,i} = y_{u,i} - p_uq_i^T$$

Em seguida, voc? ir? atualizar o conte?do dos vetores de usu?rio ($p_u$) e item ($q_i$) com base em uma taxa de aprendizado $\alpha$ e um fator de regulariza??o $\lambda$:

$$p_{u} = p_{u} + \alpha \cdot (e_{u,i} \cdot q_i - \lambda \cdot p_u)$$
$$q_{i} = q_{i} + \alpha \cdot (e_{u,i} \cdot p_u - \lambda \cdot q_i )$$

**Importante**: a atualiza??o de $p_u$ e $q_i$ deve ser feita **simultaneamente**. Assim, antes de atualizar o valor de qualquer um deles, voc? deve j? ter calculado o gradiente.

In [ ]:
def svd_sgd_optimizer(pu, qi, alpha_lr, lambda_reg, y):
    """
    Ajusta os valores pu e qi simultaneamente para aproxim?-los de y, utilizando regulariza??o
    
    ----------- Entrada -----------
       
    pu e qi: vetores np.array com os fatores latentes do usu?rio e do item, respectivamente
            - pu possui dimens?es 1 x f, onde f ? o n?mero de fatores latentes
            - qi possui dimens?es 1 x f, onde f ? o n?mero de fatores latentes
        
    alpha_lr: n?mero real representando a taxa de aprendizado
    
    lambda_reg: n?mero real representando o fator de regularizacao
    
    y: nota que o usu?rio atribuiu ao item
    
    ----------- Sa?da -----------
    
    new_pu: novo vetor de fatores latentes do usu?rio
    
    new_qi: novo vetor de fatores latentes do item
    """
    
    new_pu = np.zeros(len(pu)) # Inicializa o novo vetor do usuario
    new_qi = np.zeros(len(qi)) # Inicializa o novo vetor do item
                   
    ###########################################################################
    ######################### COMPLETE O C?DIGO AQUI  #########################
    # Instru??es: voc? deve calcular os novos valores para os vetores de fator
    # latente do item e do usu?rio
    #
    # Importante: lembre de atualizar os dois vetores simultaneamentes, ou seja,
    # n?o use o valor alterado de um dos vetores para atualizar o outro
    #
    
    e_ui = y - np.dot(pu, qi.T)
    new_pu = pu + alpha_lr * (np.dot(e_ui, qi) - lambda_reg * pu)
    new_qi = qi + alpha_lr * (np.dot(e_ui, pu) - lambda_reg * qi)
    
    ###########################################################################
    
    return new_pu, new_qi


# Verifica a implementacao

# Inicializa vetores U e I aleatoriamente
U = np.array([
    [ 0.9,  0.6,  0.5], # Usu?rio 0
    [-0.2,  0.3, -0.5], # Usu?rio 1
    [ 2.8,  1.6, -0.1], # Usu?rio 2    
])

I = np.array([
    [ 0.2,  9.3, -8.3], # Item 0
    [-0.4, -2.3,  5.4], # Item 1
    [-1.1, -0.9,  7.6], # Item 2    
])

print("Se seu c?digo estiver correto, o novo vetor do usu?rio ser? aprox.: [ 0.7775 -0.0998  2.1411]")
new_pu, new_qi = svd_sgd_optimizer(pu=U[0], qi=I[1], alpha_lr=0.10, lambda_reg=0.01, y=4.0)
print("O novo vetor do usu?rio ? {}\n".format(new_pu))

print("Se seu c?digo estiver correto, o novo vetor do item ser? aprox.: [-0.66212 -0.64924  7.56924]")
new_pu, new_qi = svd_sgd_optimizer(pu=U[2], qi=I[2], alpha_lr=0.02, lambda_reg=0.1, y=2.5)
print("O novo vetor do item ? {}\n".format(new_qi))

print("Se seu c?digo estiver correto, o novo vetor do usu?rio ser? aprox.: [-0.59275 -0.0225   2.21825]")
new_pu, new_qi = svd_sgd_optimizer(pu=U[1], qi=I[2], alpha_lr=0.05, lambda_reg=0.05, y=3.3)
print("O novo vetor do usu?rio ? {}\n".format(new_pu))

print("Se seu c?digo estiver correto, o novo vetor do item ser? aprox.: [-0.36054 -2.25336  5.3657 ]")
new_pu, new_qi = svd_sgd_optimizer(pu=U[0], qi=I[1], alpha_lr=0.01, lambda_reg=1, y=4.9)
print("O novo vetor do item ? {}\n".format(new_qi))

#### **Predi??o de nota atrav?s do SVD**

Em seguida, iremos implementar a fun??o `svd()`, que ir? utilizar sua fun??o implementada na c?lula anterior.

Esta fun??o ir?, ao longo de uma s?rie de itera??es, ajustar os fatores latentes de usu?rio e item para todas as intera??es e realizar a predi??o das notas para um conjunto de usu?rios-itens.

In [ ]:
def svd(interactions_matrix, user_item_targets, n_factors=100, n_epochs=10, alpha_lr=0.01, lambda_reg=0.01, verbose=True):
    
    # Inicializa os fatores latentes aleatoriamente
    np.random.seed(0)
    I = np.random.rand(interactions_matrix.shape[0], n_factors)
    np.random.seed(0)
    U = np.random.rand(interactions_matrix.shape[1], n_factors)
    
    ############################ TREINAMENTO ############################
    
    # Ao longo de N itera??es...
    for epc in range(n_epochs):
        # Para cada intera??o...
        all_interactions = interactions_matrix.nonzero()
        for itr, (item, user) in enumerate(zip(*all_interactions), start=1):
            if itr % 100 == 0 and verbose:
                print("SVD - Itera??o: {:02d}/{:02d}  |  Intera??o: {:05d}/{:05d}".format(epc+1, n_epochs, itr, len(all_interactions[0])), end='\r', flush=True)
            # Recupera a nota
            rating = interactions_matrix[item, user]
            # Calcula novos vetores de fatores latentes
            new_pu, new_qi = svd_sgd_optimizer(U[user], I[item], alpha_lr, lambda_reg, rating)
            # Atualiza vetores de fatores latentes
            U[user] = new_pu
            I[item] = new_qi
        if verbose:
            print("SVD - Itera??o: {:02d}/{:02d}  |  Intera??o: {:05d}/{:05d}".format(epc+1, n_epochs, itr, len(all_interactions[0])), end='\r', flush=True)
       
    ############################ PREDICAO DA NOTA ############################
    
    all_ratings = np.clip(np.dot(U, I.T), a_min=1, a_max=5) # Delimita a predi??o dentro do intervalo de notas
    ratings = all_ratings[user_item_targets[:,0], user_item_targets[:,1]]
    
    return ratings

---
## Parte 4: M?tricas de avalia??o

Com nossos algoritmos implementados, estamos prontos para recomendar. Entretanto, como podemos saber qual algoritmo foi melhor? Para isso, s?o usadas m?tricas de avalia??o.

Iremos implementar duas fam?lias diferentes de m?tricas avaliativas: aquelas que medem a qualidade de um algoritmo para prever notas, e as que medem a qualidade de um algoritmo em recomendar uma lista de itens de forma ordenada.

### <center> **Predi??o de Nota** </center>

As duas m?tricas mais comuns quando temos algoritmos que prevem uma nota s?o o **Erro M?dio Absoluto (MAE)** e a **Raiz do Erro M?dio Quadr?tico (RMSE)**. Ambos s?o medidas de erro, assim, quanto **menor** o valor, **melhor** o m?todo.

Voc? dever? implementar a fun??o `rmse_mae()`, respons?vel por calcular cada uma das m?tricas usando as seguintes f?rmulas:

$$\text{RMSE}(y, \hat{y}) = \sqrt{\frac{1}{n} \times \sum_{i=1}^{n}{(y_i - \hat{y}_i)^2}}$$

$$\text{MAE}(y, \hat{y}) = \frac{1}{n} \times \sum_{i=1}^{n}{\lvert y_i - \hat{y}_i \rvert}$$

A fun??o ir? receber dois vetores como entrada: um com as notas reais ($y$, na vari?vel `real`) e outro com as notas previstas ($\hat{y}$, na vari?vel `pred`). O retorno ser? dois n?meros reais, um para cada m?trica, variando de $0$ a $\infty$.

In [ ]:
import numpy as np

def rmse_mae(real, pred):
    """
    Calcula a Raiz do Erro M?dio Quadr?tico e o Erro M?dio Absoluto entre os valores de real e pred
    
    ----------- Entrada -----------
       
    real: vetor np.array com as notas reais
    
    pred: vetor np.array com as notas previstas
    
    ----------- Sa?da -----------
            
    rmse_score: Raiz do Erro M?dio Quadr?tico da predi??o
    
    mae_score: Erro M?dio Absoluto da predi??o
    """
    
    mae_score = 0.0 # Inicializa o MAE
    rmse_score = 0.0 # Inicializa o RMSE
    
    ###########################################################################
    ######################### COMPLETE O C?DIGO AQUI  #########################
    # Instru??es: voc? deve calcular o MAE entre as notas reais, em real, e
    # previstas, em pred
    #    
    
    rmse_score = np.sum((real - pred) ** 2) / real.shape[0]
    rmse_score = np.sqrt(rmse_score)

    mae_score = np.sum(np.abs(real - pred)) / real.shape[0]    
    
    ###########################################################################
    
    return rmse_score, mae_score

# Verifica a implementacao
print("Se seu c?digo estiver correto, o MAE ser? 1.333 e o RMSE ser? 1.6330")
rmse_score, mae_score = rmse_mae(np.array([1, 5, 3]), np.array([1, 3, 1]))
print("O MAE ? {:.4f} e o RMSE ? {:.4f}\n".format(mae_score, rmse_score))

print("Se seu c?digo estiver correto, o MAE ser? 2.5000 e o RMSE ser? 2.7386")
rmse_score, mae_score = rmse_mae(np.array([5, 5, 5, 5]), np.array([1, 2, 3, 4]))
print("O MAE ? {:.4f} e o RMSE ? {:.4f}\n".format(mae_score, rmse_score))

print("Se seu c?digo estiver correto, o MAE ser? 0.6200 e o RMSE ser? 0.8379")
rmse_score, mae_score = rmse_mae(np.array([1.5, 2.6, 4.9, 3.4, 2.8]), np.array([1.1, 4.3, 5.0, 3.1, 2.2]))
print("O MAE ? {:.4f} e o RMSE ? {:.4f}\n".format(mae_score, rmse_score))

### <center> **Ranqueamento Top-N** </center>

Das m?tricas que avaliam a qualidade do ranqueamento top-N de um algoritmo, implementaremos 4 delas: Precis?o, Revoca??o, F-Medida e NDCG, sendo que a ?ltima ser? implementada por voc?!

Todas as m?tricas consumir?o um dataframe de recomenda??es que cont?m tr?s colunas: usu?rio-alvo, item recomendado e o rank da recomenda??o (que ir? variar de 1 a N). Para implementarmos as m?tricas, iremos gerar uma base falsa de intera??es para usar como teste, e uma base falsa de recomenda??es para usar como a recomenda??o top-N.

In [ ]:
# Monta a base de intera??es de teste
fake_real = pd.DataFrame(
    [[0, 0], [0, 1], [1, 1], [1, 10], [1, 100], [2, 11], [2, 22], [3, 100], [4, 0], [4, 20], [4, 100], [4, 400]],
    columns=['id_usuario', 'id_item']
)
print("Intera??es de teste")
display(fake_real)

# Monta as recomenda??es
fake_pred = pd.DataFrame(
    [[0, 0, 1], [0, 1, 2], [0, 2, 3], 
     [1, 0, 1], [1, 10, 2], [1, 20, 3],
     [2, 10, 1], [2, 20, 2], [2, 22, 3], 
     [3, 1, 1], [3, 11, 2], [3, 400, 3],
     [4, 0, 1], [4, 20, 2], [4, 400, 3]],
    columns=['id_usuario', 'id_item', 'rank']
)
print("Recomenda??es")
display(fake_pred)

### **Precis?o, Revoca??o e F-Medida**

Para avaliar a qualidade dos itens selecionados por cada algoritmo, podemos comparar o conte?do das listas top-$N$ com os itens consumidos pelos usu?rios numa base de teste, calculando m?tricas avaliativas.

A precis?o busca calcular quantos acertos houveram dentro de tudo que foi recomendado, sendo descrita pela f?rmula:

$$\text{Prec} = \frac{\text{Qtde de recomenda??es corretas}}{\text{Qtde de itens recomendados}} = \frac{\text{Qtde de recomenda??es corretas}}{\text{Qtde de usu?rios} \times N}$$

A revoca??o busca verificar quanto foi acertado de tudo aquilo que realmente poderia ser acertado. ? calculada atrav?s da f?rmula:

$$\text{Rec} = \frac{\text{Qtde de recomenda??es corretas}}{\text{Qtde de itens consumidos pelos usu?rios}} = \frac{\text{Qtde de recomenda??es corretas}}{\text{Tamanho da base de teste}}$$

Por fim, a F-Medida ? uma forma de equilibrar a Precis?o e a Revoca??o em uma m?trica ?nica. ? calculada como:

$$\text{F1} = 2 \times \frac{\text{Prec} \times \text{Rec}}{\text{Prec} + \text{Rec}}$$

Voc? deve implementar a fun??o `precision_recall_f1()`, que recebe como entrada um conjunto de intera??es real e um previsto, e retorna a precis?o, a revoca??o e a f-medida.

In [ ]:
import pandas as pd

def precision_recall_f1(real, pred):
    """
    Calcula a precis?o, revoca??o e f-medida da recomenda??o
    
    ----------- Entrada -----------
       
    real: DataFrame com as intera??es reais, de forma similar a interactions_test
    
    pred: DataFrame com as recomenda??es, no formato (usuario, item, rank)
    
    ----------- Sa?da -----------
    
    precision_score: N?mero real contendo a precis?o da recomenda??o
    
    recall_score: N?mero real contendo a revoca??o da recomenda??o
    
    f1_score: N?mero real contendo a f-medida da recomenda??o, que dever? ser 0 se precision e recall tamb?m forem 0
    
    """
    
    precision_score = 0.0 # Inicializa a precis?o
    recall_score = 0.0 # Inicializa a revoca??o
    f1_score = 0.0 # Inicializa a f-medida
    
    # encontra as recomenda??es feitas corretamente
    hits = real.merge(pred, on=['id_usuario', 'id_item'], how='inner')

    ###########################################################################
    ######################### COMPLETE O C?DIGO AQUI  #########################
    # Instru??es: voc? deve calcular a precis?o, a revoca??o e a f-medida
    #  da recomenda??o
    #
    
    precision_score = hits.shape[0] / pred.shape[0]
    recall_score = hits.shape[0] / real.shape[0] 
    f1_score = 2 * (precision_score * recall_score) / (precision_score + recall_score)

    ###########################################################################
    
    return precision_score, recall_score, f1_score

    
# Verifica a implementacao
prec, rec, f1 = precision_recall_f1(fake_real, fake_pred)

print("Se seu c?digo estiver correto, a precis?o da recomenda??o ser? aprox.: 0.4667")
print("A precis?o da recomenda??o ?: {:.4f}\n".format(prec))

print("Se seu c?digo estiver correto, a revoca??o da recomenda??o ser? aprox.: 0.5833")
print("A revoca??o da recomenda??o ?: {:.4f}\n".format(rec))

print("Se seu c?digo estiver correto, a f-medida da recomenda??o ser? aprox.: 0.5185")
print("A f-medida da recomenda??o ?: {:.4f}\n".format(f1))

### **Ganho Cumulativo Descontado Normalizado (NDCG)**

O NDCG ? uma m?trica que busca avaliar a qualidade do ordenamento, ponderando as recomenda??es corretas de acordo com a sua posi??o no rank e usando um decaimento logaritmico.

? calculado como:

$$NDCG = \frac{DCG}{IDCG}$$

Onde DCG representa o ganho n?o normalizado, calculado por:

$$DCG = \sum_{u \in U}{\sum_{i \in R_u}{\frac{rel(i, u)}{log_2(rank_i + 1)}}}$$

Onde $U$ ? o conjunto de usu?rios, $R_u$ s?o os itens recomendados para o usu?rio, $rank_i$ ? a posi??o do item na recomenda??o e $rel(i, u)$ ?

$$rel(i, u) = 0 \text{ se o usu?rio } u \text{ N?O interagiu com o item } i \text{, e } 1 \text{ caso contr?rio}$$

J? IDCG representa o DCG ideal, ou seja, aquele que seria obtido numa recomenda??o perfeita. ? dado por:

$$IDCG = \sum_{u \in U}{\sum_{k = 1}^{T_{u_N}}{\frac{1}{log_2(rank_i + 1)}}}$$

Id?ntico ao DCG, mas com a suposi??o que todos os itens foram encontrados e ordenados corretamente. 

**IMPORTANTE**: por procurar o ganho da recomenda??o ideal, ? importante que o IDCG considere apenas as primeiras $N$ posi??es. Assim, se um usu?rio consumiu mais itens que $N$ na base de teste, o IDCG deve ser calculado apenas para $N$ itens, descartando os demais, assim ? poss?vel vermos como seria uma recomenda??o perfeita.

Voc? deve implementar a fun??o `ndgc()`, que recebe como entrada um conjunto de intera??es real, um previsto (tal como a fun??o `precision_recall_f1`) e o $N$ da recomenda??o, e retorna o DCG, o IDCG e o NDCG.

**Dica**: observe na fun??o `precision_recall_f1()` como foi feita a descoberta das recomenda??es corretas. Isso pode te ajudar na implementa??o da fun??o

In [ ]:
def ndcg(real, pred, top_n):
    """
    Calcula o DCG, IDCG e NDCG da recomenda??o
    
    ----------- Entrada -----------
       
    real: DataFrame com as intera??es reais, de forma similar a interactions_test
    
    pred: DataFrame com as recomenda??es, no formato (usuario, item, rank)
    
    top_n: valor de N para a recomenda??o top-N
    
    ----------- Sa?da -----------
    
    dcg_score: N?mero real contendo o ganho cumulativo descontado (DCG) da recomenda??o
    
    idcg_score: N?mero real contendo o ganho cumulativo descontado ideal (IDCG) da recomenda??o
    
    ndcg_score: N?mero real contendo o ganho cumulativo descontado normalizado (NDCG) da recomenda??o
    
    """
    
    dcg_score = 0.0   # Inicializa o DCG
    idcg_score = 0.0  # Inicializa o IDCG
    ndcg_score = 0.0  # Inicializa o NDCG
    
    ###########################################################################
    ######################### COMPLETE O C?DIGO AQUI  #########################
    # Instru??es: voc? deve calcular o DCG, o IDCG e o NDCG da recomenda??o
    #
    
    for user in set(real['id_usuario']):
        # Calcula o dcg
        items_real = real[real['id_usuario'] == user]['id_item'].values
        items_pred = pred[pred['id_usuario'] == user]['id_item'].values
        rel = np.isin(items_pred, items_real)
        dcg_score += np.sum(rel / np.log2(np.arange(2, rel.size + 2))) 
        
        # Calcula o idcg
        for i in range(1, top_n + 1):
            if i > len(items_real):
                break    
            
            idcg_score += 1 / np.log2(i + 1)

    # Calcula o ndcg
    ndcg_score = dcg_score / idcg_score

    ###########################################################################
    
    return dcg_score, idcg_score, ndcg_score


# Verifica a implementacao
dcg_score, idcg_score, ndcg_score = ndcg(fake_real, fake_pred, top_n=3)

print("Se seu c?digo estiver correto, o DCG da recomenda??o ser? aprox.: 4.8928")
print("O DCG da recomenda??o ?: {:.4f}\n".format(dcg_score))

print("Se seu c?digo estiver correto, o IDCG da recomenda??o ser? aprox.: 8.5237")
print("O IDCG da recomenda??o ?: {:.4f}\n".format(idcg_score))

print("Se seu c?digo estiver correto, o NDCG da recomenda??o ser? aprox.: 0.5740")
print("O NDCG da recomenda??o ?: {:.4f}\n".format(ndcg_score))

---
## Parte 5: Ajuste de par?metros

Nesta etapa, iremos testar diversos valores para os hiperpar?metros dos modelos, para descobrir, atrav?s de uma busca em grande sobre a valida??o, qual a melhor combina??o de par?metros. Os par?metros selecionados ser?o aqueles que minimizarem a m?trica RMSE.

In [ ]:
# Recupera os pares usuario-item alvo e as notas reais
user_item_targets = interactions_val[['id_usuario', 'id_item']].values
real_ratings = interactions_val['nota'].values

# --- KNN
print("---------- KNN ----------")
knn_best_params, knn_best_rmse = None, np.inf
for k in [3, 5, 10, 15]:
    _, pred_ratings = knn(interactions_matrix, user_item_targets, k=k)
    pred_rmse, _ = rmse_mae(real_ratings, pred_ratings)
    print('\tk = {}  |  RMSE: {:.5f}'.format(k, pred_rmse))
    if pred_rmse < knn_best_rmse:        
        knn_best_rmse = pred_rmse
        knn_best_params = {'k': k}
print("Melhores par?metros encontrados para o KNN: {}\n".format(knn_best_params))

# --- SVD
print("---------- SVD ----------")
svd_best_params, svd_best_rmse = None, np.inf
for n_factors in [50, 100]:
    for alpha_lr in [0.01, 0.001]:
        for lambda_reg in [0.01, 0.001]:        
            pred_ratings = svd(interactions_matrix, user_item_targets, n_factors=n_factors, alpha_lr=alpha_lr, lambda_reg=lambda_reg, verbose=False)
            rmse_score, _ = rmse_mae(real_ratings, pred_ratings)
            print('\tn_factors = {}  |  alpha_lr = {}  |  lambda_reg = {} |  RMSE: {:.5f}'.format(n_factors, alpha_lr, lambda_reg, pred_rmse))            
            if pred_rmse < svd_best_rmse:
                svd_best_rmse = pred_rmse
                svd_best_params = {'n_factors': n_factors, 'alpha_lr': alpha_lr, 'lambda_reg': lambda_reg}
print("Melhores par?metros encontrados para o SVD: {}\n".format(svd_best_params))

---
## Parte 6: Avalia??o dos algoritmos de recomenda??o

Finalmente, podemos executar nossos algoritmos sobre a base de teste e verificar o resultado final.

#### **Junta bases de treinamento e valida??o**

Para enriquecer o treinamento, iremos juntar nossa base de valida??o com a de treinamento.

In [ ]:
interactions_all = pd.concat([interactions_train, interactions_val]).copy()

interactions_matrix = np.zeros((n_items, n_users))
interactions_matrix[interactions_all['id_item'], interactions_all['id_usuario']] = interactions_all['nota'].values

#### **Tarefa de predi??o de nota**

Nesta tarefa, os algoritmos ir?o prever as notas para um conjunto de pares usu?rio-item, tendo suas predi??es avaliadas pelo MAE e o RMSE.

In [ ]:
# Recupera pares usuario-item
rating_pred_user_item_targets = interactions_test[['id_usuario', 'id_item']].values
real_rating_pred = interactions_test['nota'].values

# Preve as notas
print('Prevendo as notas...')
_, knn_rating_pred = knn(interactions_matrix, rating_pred_user_item_targets, **knn_best_params)
svd_rating_pred = svd(interactions_matrix, rating_pred_user_item_targets, **svd_best_params)

# Calcula os resultados
print('\nCalculando m?tricas...')
knn_rmse, knn_mae = rmse_mae(knn_rating_pred, real_rating_pred)
svd_rmse, svd_mae = rmse_mae(svd_rating_pred, real_rating_pred)

print('OK!')

#### **Tarefa de ranqueamento top-N**

Para recomendar utilizando os algoritmos implementados, ter?amos que prever as notas que os usu?rios-alvo dariam para TODOS os itens n?o consumidos, o que pode ser extremamente custoso. Em sistemas reais, os algoritmos estariam implementados de forma mais otimizada, al?m de outras estrat?gias que poderiam ser adotadas, como uma pr?-etapa de clusteriza??o ou uso de recomendadores espec?ficos.

Para que o notebook rode em tempo vi?vel e n?o demande um computador com alta disponibilidade de recursos, iremos gerar uma ranking top-20 apenas para os 10 usu?rios na base de teste que mais interagiram com itens, avaliando atrav?s da Precis?o, Revoca??o, F-Medida e NDCG.

In [ ]:
# Recupera os usuarios-alvo
NUM_TOP_N_USERS = 10
top_n_users = interactions_test['id_usuario'].value_counts().sort_values(ascending=False).index.values[:NUM_TOP_N_USERS]

# Monta um dataframe com TODOS os pares usuario-item usando os usuarios-alvo
top_n_user_item_targets = pd.DataFrame([], columns=['id_usuario', 'id_item']).astype(int)
for tnu in top_n_users:    
    not_consumed_items = np.where(interactions_matrix[:, tnu]==0)[0]

    top_n_user_df = pd.DataFrame(
        np.vstack([np.repeat(tnu, len(not_consumed_items)), not_consumed_items]).T,
        columns=['id_usuario', 'id_item']
    ).astype(int)    

    top_n_user_item_targets = pd.concat([top_n_user_item_targets, top_n_user_df])

# Preve as notas atribuidas pelos usuarios para todos os itens nao consumidos
print('Prevendo as notas...')
_, knn_ratings = knn(interactions_matrix, top_n_user_item_targets[['id_usuario', 'id_item']].values)
svd_ratings = svd(interactions_matrix, top_n_user_item_targets[['id_usuario', 'id_item']].values)
top_n_user_item_targets['nota-knn'] = knn_ratings
top_n_user_item_targets['nota-svd'] = svd_ratings

# Gera a recomendacao top-20
print('\nMontando listas top-N...')
TOP_N = 20

knn_top_n = top_n_user_item_targets.sort_values('nota-knn', ascending=False).groupby('id_usuario').head(TOP_N)[['id_usuario', 'id_item']].sort_values('id_usuario')
knn_top_n['rank'] = np.tile(np.arange(1, TOP_N+1), NUM_TOP_N_USERS)

svd_top_n = top_n_user_item_targets.sort_values('nota-svd', ascending=False).groupby('id_usuario').head(TOP_N)[['id_usuario', 'id_item']].sort_values('id_usuario')
svd_top_n['rank'] = np.tile(np.arange(1, TOP_N+1), NUM_TOP_N_USERS)

real_top_n = interactions_test[['id_usuario', 'id_item']] # Interacoes de teste

# Calcula as metricas
print('Calculando m?tricas...')
knn_prec, knn_rec, knn_f1 = precision_recall_f1(real_top_n, knn_top_n)
_, _, knn_ndcg = ndcg(real_top_n, knn_top_n, TOP_N)

svd_prec, svd_rec, svd_f1 = precision_recall_f1(real_top_n, svd_top_n)
_, _, svd_ndcg = ndcg(real_top_n, svd_top_n, TOP_N)

print('OK!')

#### **Resultados finais**

Os resultados obtidos s?o bastante interessantes. Ainda que em aplica??es reais o SVD geralmente obtenha resultados superiores ao KNN, neste experimento, o algoritmo baseado em vizinhan?a se saiu superior a fatora??o de matriz em todas as m?tricas, o que mostra que est? sendo melhor em aproximar as notas relacionadas aos itens da base de teste e ainda aprender uma boa ordem para os itens que o usu?rio gostaria de consumir. Ainda assim, n?o podemos afirmar que um m?todo ? superior ao outro porque estamos num cen?rio pouco condizente com a realidade. Fizemos muitas amostragens dos dados e temos uma base com poucos clientes e baixa signific?ncia estat?stica.

In [ ]:
results = pd.DataFrame([
    [knn_mae, svd_mae],
    [knn_rmse, svd_rmse],
    [knn_prec, svd_prec],
    [knn_rec, svd_rec],
    [knn_f1, svd_f1],
    [knn_ndcg, svd_ndcg]
], columns=['KNN', 'SVD'], index=['MAE', 'RMSE', 'Prec', 'Rec', 'F1', 'NDCG'])
display(results)